# 06_特征选择方法-信息增益

## Notebook 使用说明

建议先阅读理论，再从上到下运行全部单元；使用 Python 3.11–3.13 内核。数据与辅助模块均来自本文件夹，安装依赖后无需联网获取案例数据。

本篇使用代码生成的模拟数据或 scikit-learn 随包附带的小数据集，无需外部数据文件。

原文参考图及静态数值仅用于对照；重跑后的代码输出为本次实验结果。依赖版本与验证记录见根目录 `README.md` 和 `reports/`。

In [1]:
# 从当前目录向上寻找本套 Notebook 的根目录，移动整个文件夹后仍可运行。
from pathlib import Path
import os, sys, tempfile, random
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebook_support.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从这套 Notebook 文件夹内启动 Jupyter，并保留 notebook_support.py。")
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "ml_notebooks_matplotlib"))
from notebook_support import DATA_DIR, load_local_housing, load_local_boston, load_local_mnist, load_creditcard
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
random.seed(42)
np.random.seed(42)
# 按实际安装的字体选择，兼容 macOS / Windows / Linux。
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
CJK_FONTS = [f for f in ["Arial Unicode MS", "PingFang SC", "Heiti TC", "Microsoft YaHei", "SimHei", "Noto Sans CJK SC"]
             if f in _available_fonts] + ["DejaVu Sans"]
plt.rcParams.update({"font.sans-serif": CJK_FONTS, "axes.unicode_minus": False,
                     "figure.dpi": 90, "figure.max_open_warning": 30})
%matplotlib inline


今天，咱们聊聊特征选择方法中的**信息增益**\~

#### 什么是特征选择？

在机器学习的学习中，我们经常有一大堆数据特征（比如年龄、性别、收入、爱好等等），但并不是每个特征都对预测结果有用。  
**特征选择**就是从中挑选出**最有用的特征**，帮助我们做出更准确的预测，还能减少计算量，避免过拟合。

#### 什么是信息增益？

先想象一个场景：

你正在玩“猜动物”的游戏，一开始你啥都不知道，要靠提问题来缩小范围。

如果你问：“这动物是猫吗？”

答案要么是“是”，要么“不是”，但这个问题可能不太有帮助。

但如果你问：“这动物是哺乳动物吗？”

这个问题一下就能把所有动物分成两大类，缩小了很多范围，对你猜对答案特别有帮助。

这个问题带来的帮助，就是信息增益！

### 信息增益

**信息增益 = 提出某个特征后，我们对分类结果了解得多了多少。**

换句话说，就是：某个特征到底有多“有用” - 能不能帮我们把不同类别更好地区分开？

1. **先看整体：**假设你有一堆样本，混合了很多类别（比如好瓜/坏瓜）。这时候你是不知道谁是谁的，信息很混乱。
2. **看某个特征（比如颜色）**：你按照颜色把瓜分组：绿色的、白色的、黑色的…… 每一组里面再看“好瓜”和“坏瓜”的分布，是不是更清晰了？
3. **如果分完之后，组内变得“更纯了”**（比如绿色的几乎都是好瓜），那说明这个特征很有用，**信息增益就高**！
4. **如果分完之后，组内还是乱七八糟的（好坏瓜都有），那说明这个特征没啥用，信息增益就低。**

总之，信息增益告诉我们：某个特征，能不能帮我们更好地区分不同的类别。越能区分，信息增益越大，就越值得选用！

## 原理详解

在分类问题中，我们希望通过选择最能区分类别的特征来构建模型。信息增益衡量的是使用某个特征进行划分后，样本纯度的提升程度，也就是**利用某个特征可以获得多少信息**，从而帮助我们判断该特征的重要性。

信息增益基于**信息论**中的熵（Entropy）概念。

### 1. 熵

熵用来度量一个数据集的“纯净度”或“不确定性”。熵越大，不确定性越高，类别分布越均匀；熵越小，数据越纯净，类别越集中。

假设数据集 $D$ 有 $K$ 个类别，类别 $C_k$ 在数据集中所占的比例为 $p_k$，则熵定义为：


$$
H(D) = - \sum_{k=1}^{K} p_k \log_2 p_k
$$


其中：

- $p_k = \frac{|D_k|}{|D|}$，$|D_k|$ 表示类别 $C_k$ 的样本数量
- 约定 $0 \log 0 = 0$

### 2. 条件熵

当利用特征 $A$ 把数据集 $D$ 划分成若干子集时，数据集的熵会变化。

假设特征 $A$ 有 $V$ 个可能的取值（或离散区间）：


$$
A = \{a_1, a_2, \dots, a_V\}
$$


利用 $A$ 的值将数据集 $D$ 划分为 $V$ 个子集：


$$
D_j = \{x \in D \mid x \text{在特征} A \text{上取值为} a_j\}, \quad j=1,2,\dots,V
$$


则特征 $A$ 对 $D$ 的条件熵为：


$$
H(D|A) = \sum_{j=1}^V \frac{|D_j|}{|D|} H(D_j)
$$


即特征划分后的加权熵，权重为每个子集的比例。

### 3. 信息增益

信息增益定义为划分前后熵的减少量，即：

- $IG(D, A) = H(D) - H(D|A)$
- $H(D)$ 是划分前的熵（总熵）
- $H(D|A)$ 是划分后的条件熵（特征 $A$ 划分后的熵）

信息增益度量的是使用特征 $A$ 划分后“信息”的增加量，信息增益越大，说明该特征越能有效减少不确定性，更有助于区分类别。

## 核心公式

假设数据集 $D$ 有两类（比如好瓜和坏瓜），分别用 $C_1$ 和 $C_2$ 表示，数据集大小为 $|D|$。

1. 计算整体熵：


$$
p_1 = \frac{|D_1|}{|D|}, \quad p_2 = \frac{|D_2|}{|D|}
$$


$$
H(D) = -p_1 \log_2 p_1 - p_2 \log_2 p_2
$$


1. 按特征 $A$ 取值分组，比如 $A$ 有 $V$ 个取值：


$$
D = D_1 \cup D_2 \cup \dots \cup D_V
$$


每个子集内的类别概率为 $p_{k|j} = \frac{|D_{j,k}|}{|D_j|}$，其中 $D_{j,k}$ 表示第 $j$ 个子集中类别 $k$ 的样本数。

1. 计算每个子集的熵：


$$
H(D_j) = - \sum_{k=1}^K p_{k|j} \log_2 p_{k|j}
$$


1. 计算条件熵：


$$
H(D|A) = \sum_{j=1}^V \frac{|D_j|}{|D|} H(D_j)
$$


1. 计算信息增益：


$$
IG(D, A) = H(D) - H(D|A)
$$


## 算法流程

1. 准备数据：输入带标签的数据集 $D$，确定类别数量 $K$。
2. 计算总体熵 $H(D)$：统计每个类别的样本数，计算总体熵。
3. **遍历每个特征** $A$**：**

   - 按特征 $A$ 的每个取值 $a_j$ 将数据集划分为子集 $D_j$
   - 计算每个子集的类别分布及熵 $H(D_j)$
   - 计算条件熵 $H(D|A)$
   - 计算信息增益 $IG(D, A) = H(D) - H(D|A)$
4. 比较所有特征的信息增益：选择信息增益最高的特征作为分裂依据。

信息增益从熵的角度出发，量化了某个特征对数据集类别划分纯度的提升程度，计算公式：


$$
\boxed{IG(D, A) = H(D) - \sum_{j=1}^V \frac{|D_j|}{|D|} H(D_j)}
$$


利用它可以有效地做特征选择，找到最有助于分类的特征。

## 完整案例

假设我们有一个关于“是否喜欢某种水果”的数据集，特征包括颜色、大小、甜度等，我们想通过信息增益来找出最有用的特征，用于判断“喜欢”与“否”。

### 1. 数据集

In [2]:
import pandas as pd
import numpy as np

我们创建一个简单的水果数据集：

| 颜色(Color) | 大小(Size) | 甜度(Sweet) | 喜欢(Liked) |
|-|-|-|-|
| 绿色 | 大 | 甜 | 是 |
| 绿色 | 大 | 不甜 | 是 |
| 黄色 | 大 | 甜 | 否 |
| 黄色 | 小 | 甜 | 否 |
| 黄色 | 小 | 不甜 | 是 |
| 绿色 | 小 | 甜 | 否 |
| 绿色 | 大 | 不甜 | 是 |

In [3]:
# 构造数据
data = {
    'Color': ['Green', 'Green', 'Yellow', 'Yellow', 'Yellow', 'Green', 'Green'],
    'Size': ['Large', 'Large', 'Large', 'Small', 'Small', 'Small', 'Large'],
    'Sweet': ['Sweet', 'Not Sweet', 'Sweet', 'Sweet', 'Not Sweet', 'Sweet', 'Not Sweet'],
    'Liked': ['Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
print(df)

    Color   Size      Sweet Liked
0   Green  Large      Sweet   Yes
1   Green  Large  Not Sweet   Yes
2  Yellow  Large      Sweet    No
3  Yellow  Small      Sweet    No
4  Yellow  Small  Not Sweet   Yes
5   Green  Small      Sweet    No
6   Green  Large  Not Sweet   Yes


### 2. 定义计算熵的函数

In [4]:
def entropy(target_col):
    """
    计算标签列的熵
    """
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = 0
    for i in range(len(elements)):
        p = counts[i] / sum(counts)
        entropy_val -= p * np.log2(p)
    return entropy_val

计算给定标签列的熵，用于衡量整体的纯净度。

### 3. 计算条件熵

In [5]:
def conditional_entropy(data, feature_col, target_col):
    """
    计算给定特征的条件熵 H(target | feature)
    """
    elements, counts = np.unique(data[feature_col], return_counts=True)
    cond_entropy = 0
    for i in range(len(elements)):
        subset = data[data[feature_col] == elements[i]]
        p = counts[i] / sum(counts)
        ent = entropy(subset[target_col])
        cond_entropy += p * ent
    return cond_entropy

利用特征划分数据集，计算划分后的加权熵。

### 4. 计算信息增益

In [6]:
def info_gain(data, feature_col, target_col):
    """
    计算信息增益 IG = H(target) - H(target | feature)
    """
    total_entropy = entropy(data[target_col])
    cond_entropy = conditional_entropy(data, feature_col, target_col)
    return total_entropy - cond_entropy

### 5. 计算所有特征的信息增益

In [7]:
target_col = 'Liked'
features = ['Color', 'Size', 'Sweet']

for feature in features:
    ig = info_gain(df, feature, target_col)
    print(f"Feature: {feature}, Information Gain: {ig:.4f}")

Feature: Color, Information Gain: 0.1281
Feature: Size, Information Gain: 0.1281
Feature: Sweet, Information Gain: 0.5216


### 3.6 输出结果示例

原文参考输出或命令说明（无需在 Python 内核执行）：

```text
Feature: Color, Information Gain: 0.1281
Feature: Size, Information Gain: 0.1281
Feature: Sweet, Information Gain: 0.5216
```

“Sweet”特征的信息增益最大，说明它在区分类别“Liked”上最有用。

**熵函数**衡量数据纯度。标签列纯度越高，熵越小，纯度越低，熵越大。

**条件熵函数**计算在特征条件下的加权纯度，反映了使用该特征后剩余的不确定性。

**信息增益**为两者之差，代表通过特征减少的不确定性。

## 模型分析

### 信息增益特征选择方法的优缺点

**优点：**

信息增益作为特征选择的标准，最大优点是理论基础扎实，源自信息论，能量化特征对分类结果的不确定性减少程度。这使得它能够较为直观地评价一个特征是否有助于区分不同类别。对于离散特征来说，信息增益计算简单，效果明显，在该案例中能够快速识别出“颜色”这一关键特征，提升模型的解释性和效率。

**缺点：**

信息增益偏向于选择取值多的特征，因为更多的取值通常能划分出更多子集，造成信息熵下降较大，这种偏向可能导致过拟合。此外，信息增益本身不能直接处理连续数值特征，必须先进行离散化或划分阈值，增加了预处理复杂度。在特征数量非常大且特征类型复杂的场景下，信息增益的计算成本和效果可能不尽如人意。

### 与其他常见特征选择方法的对比

| 特征选择方法 | 优点 | 缺点 | 适用场景 |
|-|-|-|-|
| 信息增益 | 计算直观、理论基础清晰，适合离散特征 | 偏向多值特征，连续特征需离散化 | 离散特征数量适中，类别明确，需快速筛选有效特征 |
| 信息增益率 | 解决信息增益偏向多值特征的问题 | 计算较复杂，仍需离散化连续特征 | 多值特征较多，希望避免偏向，改进版信息增益 |
| 卡方检验 | 统计学方法，评估特征与类别相关性强弱 | 对样本量要求较高，数值特征需离散化 | 需要严格统计检验，样本充足，分类变量多 |
| 互信息 | 能捕获非线性关系，适合连续和离散特征 | 计算复杂，参数调优较难 | 特征与标签关系复杂，需要捕获多种相关性 |
| 方差选择 | 简单快速，去除低方差无效特征 | 无法直接判断特征与标签关系 | 预处理阶段快速筛选，减少无效特征 |
| 基于模型的重要性（如随机森林） | 结合模型训练结果，直接反映特征贡献 | 依赖模型，计算量大，结果受模型影响 | 模型训练后评估特征重要性，适合复杂数据和非线性场景 |

### 什么时候选择信息增益，什么时候考虑其他方法？

**适合使用信息增益的情况：**

数据集以离散特征为主，类别明确且相对均衡，且需要快速识别对类别区分贡献大的特征时。尤其是在构建决策树等基于信息论的模型时，信息增益能直接指导特征选择和划分。案例中水果颜色的区分就是典型场景。

**考虑使用其他特征选择方法的情况：**

- 当特征取值非常多，且信息增益可能导致偏向时，改用**信息增益率**来避免偏向。
- 如果数据中包含大量连续数值特征，且离散化困难或效果不佳，可以使用**互信息**或基于模型的重要性评分。
- 需要严格的统计意义检验时，**卡方检验**是更合适的选择。
- 在数据维度极高且有许多无效特征时，可以先用简单的方差筛选去除部分特征，再用信息增益等方法做进一步筛选。

总结来说，信息增益在特征选择中是一种经典且有效的方法，适合简单到中等复杂度的离散特征数据。面对更复杂、连续、多维的数据特征，结合其他特征选择方法或者采用集成模型的特征重要性评估，往往能取得更好的效果。根据数据类型、特征性质及任务需求灵活选择，是特征选择策略成功的关键。